In [1]:
!pip install -U \
    "transformers==4.46.3" \
    "bitsandbytes==0.43.3" \
    "peft==0.12.0" \
    "trl==0.10.1" \
    "accelerate==0.34.2" \
    "datasets==2.21.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
     

In [2]:
!pip uninstall -y bitsandbytes triton
!pip install -U bitsandbytes

Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00


In [1]:
import transformers
import bitsandbytes
import peft
import trl
import accelerate
import datasets

print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)

import pandas as pd
import numpy as np
from datasets import Dataset

from google.colab import userdata
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, TrainingArguments
import torch

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

os.environ["Gemma_HF"] = userdata.get('Gemma_HF')

df = pd.read_csv('/content/Poems_with_meaning.csv')
df.sample()

def format(example):
  return f"""You are an expert at explaining the meaning of the poem. Give an explanation of the poem in
                simple words, explaining the main idea of what the author is trying to say.
                Generate only the explanation part. Do not repeat yourself by writing the prompt given by user again.
                Do not write anything other than the explanation.

                Poem: {example['Poem']}
                Explanation: {example['Meaning']}"""

df['text'] = df.apply(format, axis=1)

dataset = Dataset.from_pandas(df[['text']], preserve_index=False)
dataset = dataset.train_test_split(test_size=0.1,seed=42)

train_dataset = dataset['train']
test_dataset = dataset['test']

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b",token = os.environ["Gemma_HF"])

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("google/gemma-2b",token = os.environ["Gemma_HF"], quantization_config=bnb_config, device_map="auto")

lora_config = LoraConfig(r=16,lora_alpha=16,lora_dropout=0.1, bias="none",target_modules=["q_proj","k_proj","v_proj","o_proj"],task_type="CAUSAL_LM")

# In this one, we apply LoRA on 4 projections

training_args = TrainingArguments(output_dir='./gemma_training_info_poem_meaning', per_device_train_batch_size=2, per_device_eval_batch_size=2,
                                  gradient_accumulation_steps=8,max_steps=320,learning_rate=2e-05,lr_scheduler_type='cosine',warmup_ratio=0.05, fp16=True,
                                  logging_steps=8,logging_first_step=True, eval_strategy='steps',eval_steps=64,save_strategy='steps',save_steps=64,
                                  load_best_model_at_end=True,metric_for_best_model='eval_loss',optim="paged_adamw_8bit",report_to="none")

trainer = SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=train_dataset, eval_dataset=test_dataset,dataset_text_field="text",
                     max_seq_length=512,args=training_args,peft_config=lora_config)

trainer.train()

output_dir = "./gemma-poem-meaning-lora-details-4projections-evalandsavesteps"

trainer.save_model(output_dir)

tokenizer.save_pretrained(output_dir)

transformers: 4.46.3
bitsandbytes: 0.50.1
peft: 0.12.0
trl: 0.10.1
accelerate: 0.34.2
datasets: 2.21.0


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Map:   0%|          | 0/134 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:407: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:412: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
/usr/local/lib/python3.13/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss
64,3.798100,3.655312
128,3.206400,3.010578
192,3.103700,2.874881
256,2.964900,2.856293
320,3.036600,2.853450


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:619: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6a8ae128-1919047223fbc69a06c3ffb4;7ea85453-799d-4a7f-9c05-52ea94942efb)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignor

('./gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer_config.json',
 './gemma-poem-meaning-lora-details-4projections-evalandsavesteps/special_tokens_map.json',
 './gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer.model',
 './gemma-poem-meaning-lora-details-4projections-evalandsavesteps/added_tokens.json',
 './gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer.json')

In [ ]:
./gemma-poem-meaning-lora-details-4projections-evalandsavesteps

In [12]:
!zip -r gemma-poem-meaning-lora-details-4projections-evalandsavesteps.zip ./gemma-poem-meaning-lora-details-4projections-evalandsavesteps

  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/ (stored 0%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/training_args.bin (deflated 53%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer.model (deflated 51%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer.json (deflated 84%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/special_tokens_map.json (deflated 76%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/README.md (deflated 66%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/adapter_config.json (deflated 53%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/tokenizer_config.json (deflated 96%)
  adding: gemma-poem-meaning-lora-details-4projections-evalandsavesteps/adapter_model.safetensors (deflated 8%)


In [3]:
model.eval()

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=256, bias=False)
            (lora_dropout): M

In [4]:
test_poem = test_dataset[0]["text"]

# Extract just the poem from the formatted example
test_poem = test_poem.split("Poem:")[1].split("\n\nExplanation:")[0]

prompt = f"""You are an expert at explaining the meaning of the poem. Give an explanation of the poem in
                simple words, explaining the main idea of what the author is trying to say.
                Generate only the explanation part. Do not repeat yourself by writing the prompt given by user again.
                Do not write anything other than the explanation.

Poem:
{test_poem}

Explanation:"""


inputs = tokenizer(prompt,return_tensors="pt").to(model.device)


with torch.no_grad():

    outputs = model.generate(**inputs,max_new_tokens=300,temperature=0.7,top_p=0.9,do_sample=True,repetition_penalty=1.1)


generated_text = tokenizer.decode(outputs[0],skip_special_tokens=True)

In [7]:
generated_text.split("Explanation:",1)[1].strip()

"Ephemeral wonder and shared reverence for nature’s grand spectacle unite a transient community in a moment of awe-struck observation.\n\nExplanation: The poet, whose work explores the intersection between humanity and nature, has chosen to capture the scene of a mass gathering of migratory birds in a field. The birds' presence here symbolizes the connection between humans and the natural world, as well as the significance of these annual events for both species. The poet conveys a sense of awe and reverence for the birds' majestic presence, drawing parallels between their scale and the scale of the cosmos. The poet also emphasizes the transitory nature of the event, suggesting that it will soon be over, but its impact will be felt for generations to come."

In [9]:
test_poem ="""When I see birches bend to left and right
Across the lines of straighter darker trees,
I like to think some boy’s been swinging them.
But swinging doesn’t bend them down to stay
As ice-storms do. Often you must have seen them
Loaded with ice a sunny winter morning
After a rain. They click upon themselves
As the breeze rises, and turn many-colored
As the stir cracks and crazes their enamel.
Soon the sun’s warmth makes them shed crystal shells
Shattering and avalanching on the snow-crust—
Such heaps of broken glass to sweep away
You'd think the inner dome of heaven had fallen.
They are dragged to the withered bracken by the load,
And they seem not to break; though once they are bowed
So low for long, they never right themselves:
You may see their trunks arching in the woods
Years afterwards, trailing their leaves on the ground
Like girls on hands and knees that throw their hair
Before them over their heads to dry in the sun.
But I was going to say when Truth broke in
With all her matter-of-fact about the ice-storm
I should prefer to have some boy bend them
As he went out and in to fetch the cows—
Some boy too far from town to learn baseball,
Whose only play was what he found himself,
Summer or winter, and could play alone.
One by one he subdued his father's trees
By riding them down over and over again
Until he took the stiffness out of them,
And not one but hung limp, not one was left
For him to conquer. He learned all there was
To learn about not launching out too soon
And so not carrying the tree away
Clear to the ground. He always kept his poise
To the top branches, climbing carefully
With the same pains you use to fill a cup
Up to the brim, and even above the brim.
Then he flung outward, feet first, with a swish,
Kicking his way down through the air to the ground.
So was I once myself a swinger of birches.
And so I dream of going back to be.
It’s when I’m weary of considerations,
And life is too much like a pathless wood
Where your face burns and tickles with the cobwebs
Broken across it, and one eye is weeping
From a twig’s having lashed across it open.
I'd like to get away from earth awhile
And then come back to it and begin over.
May no fate willfully misunderstand me
And half grant what I wish and snatch me away
Not to return. Earth’s the right place for love:
I don’t know where it's likely to go better.
I'd like to go by climbing a birch tree,
And climb black branches up a snow-white trunk
Toward heaven, till the tree could bear no more,
But dipped its top and set me down again.
That would be good both going and coming back.
One could do worse than be a swinger of birches."""




prompt = f"""You are an expert at explaining the meaning of the poem. Give an explanation of the poem in
                simple words, explaining the main idea of what the author is trying to say.
                Generate only the explanation part. Do not repeat yourself by writing the prompt given by user again.
                Do not write anything other than the explanation.

Poem:
{test_poem}

Explanation:"""


inputs = tokenizer(prompt,return_tensors="pt").to(model.device)


with torch.no_grad():

    outputs = model.generate(**inputs,max_new_tokens=300,temperature=0.7,top_p=0.9,do_sample=True,repetition_penalty=1.1)


generated_text = tokenizer.decode(outputs[0],skip_special_tokens=True)

In [11]:
generated_text.split("Explanation:",1)[1].strip()

'This poem is about how we tend to forget the things that happened in our past lives. The person who wrote this poem lived a life full of hardships, but also lots of joy. The speaker wants to live in the past because it seems simpler and less complicated. They want to believe that if they can remember the best moments of their life, they can avoid feeling sadness and pain in the present moment.'